# BTC跨资产状态策略：从单一开关到条件因子池

## tl;dr

在2016年8月至2026年8月的统一口径下，原始BTC开关的夏普为1.29，简化后的跨资产评分模型为1.34；真正明显的改善集中在2023—2026留出样本：夏普从1.55升至1.90，最大回撤从-12.6%降至-8.6%。但跨资产模型在2020—2022验证期明显落后，因此当前结论只能是“值得继续验证”，还不是稳定可交易的最终模型。

短期“BTC与黄金上涨、美元下跌、QQQ下跌”能够识别2026年8月的背离，但历史上它之后的QQQ平均收益并不更差，不能直接作为空头或清仓规则。

## Context & Methods

### 研究问题

QuantConnect原策略把BTC上升趋势直接解释为美股risk-on。本研究检验：加入QQQ自身趋势、美元、黄金和长债后，能否识别“BTC反法币上涨”，减少对QQQ的错误增配。

### Key Assumptions

- 信号只使用当时已经出现的价格，周五/当周最后一个交易日形成判断，下周执行。
- QQQ与SHY收益包含Nasdaq公开分红记录；其他资产只作为信号。
- 每次单边交易成本5个基点；策略在QQQ与SHY间配置。
- 夏普使用零无风险利率，仅用于同一数据和同一计算口径下的横向比较。
- 受Nasdaq公开接口近十年数据边界限制，2016年8月—2019年为开发期，2020—2022年为验证期，2023—2026年为留出样本。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "research.py").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_DIR))

from research import run_research

## Data

### 1. Load and validate public market data

In [ ]:
data, checks, features, weights, returns, by_period, full_sample = run_research(force_download=False)
checks

In [ ]:
coverage = pd.DataFrame({
    "first_valid": data.first_valid_index(),
    "last_valid": data.last_valid_index(),
    "non_null": data.notna().sum(),
    "missing_pct": data.isna().mean(),
})
coverage

## Results

### 2. Compare full-sample performance

`BTC gate`复刻原始思想；`BTC + QQQ confirm`加入股票自身趋势；`Regime score`加入美元与长债确认，并允许0%、50%、100%三级风险仓位。反法币仓位上限仅作为实验消融项，不作为正式候选。

In [ ]:
display_table = full_sample.copy()
for column in ["CAGR", "Volatility", "Max_drawdown", "QQQ_exposure"]:
    display_table[column] = display_table[column].map(lambda x: f"{x:.1%}")
for column in ["Sharpe_rf0", "Calmar"]:
    display_table[column] = display_table[column].map(lambda x: f"{x:.2f}")
display_table

In [ ]:
equity = (1 + returns.fillna(0)).cumprod()
ax = equity.plot(figsize=(12, 6), logy=True, linewidth=1.8)
ax.set_title("Cumulative wealth, 2016–2026 (log scale)")
ax.set_ylabel("Growth of $1")
ax.grid(alpha=0.25)
plt.show()

### 3. Inspect validation and holdout periods

In [ ]:
period_table = by_period.loc[by_period["Period"].ne("Full sample")].copy()
period_table.pivot(index="Strategy", columns="Period", values=["Sharpe_rf0", "Max_drawdown", "CAGR"]).round(3)

### 4. Test the short-term anti-fiat divergence

In [ ]:
short_term_anti_fiat = (
    features["btc_roc5"].gt(0)
    & features["gold_roc5"].gt(0)
    & features["dollar_roc5"].lt(0)
    & features["qqq_roc5"].lt(0)
)
anti_fiat_summary = pd.DataFrame({
    "days": [int(short_term_anti_fiat.sum())],
    "share_of_sessions": [short_term_anti_fiat.mean()],
    "next_5d_QQQ_return": [data.loc[features.index, "QQQ"].pct_change(5).shift(-5).loc[short_term_anti_fiat].mean()],
    "next_20d_QQQ_return": [data.loc[features.index, "QQQ"].pct_change(20).shift(-20).loc[short_term_anti_fiat].mean()],
    "all_days_next_20d_QQQ_return": [data.loc[features.index, "QQQ"].pct_change(20).shift(-20).mean()],
})
anti_fiat_summary

### 5. Block-bootstrap the holdout Sharpe improvement

In [ ]:
from research import moving_block_bootstrap_sharpe_diff

holdout = returns.loc["2023-01-01":"2026-08-24"]
moving_block_bootstrap_sharpe_diff(
    holdout["Regime score"],
    holdout["BTC gate"],
    block_size=20,
    simulations=2000,
)

### 6. Stress transaction costs

In [ ]:
from research import performance_metrics, strategy_returns

cost_rows = []
for bps in [0, 5, 10, 20]:
    cost_returns = strategy_returns(data, weights, one_way_cost_bps=bps)
    for period, start in [("Full sample", "2016-08-25"), ("Holdout", "2023-01-01")]:
        for strategy in ["BTC gate", "Regime score"]:
            metrics = performance_metrics(
                cost_returns.loc[start:"2026-08-24", strategy], weights[strategy]
            )
            cost_rows.append({"one_way_bps": bps, "period": period, "strategy": strategy, **metrics})
pd.DataFrame(cost_rows)[["one_way_bps", "period", "strategy", "CAGR", "Sharpe_rf0", "Max_drawdown"]].round(3)

### 7. Current-state readout

In [ ]:
latest = features.dropna().iloc[-1]
pd.Series({
    "date": features.dropna().index[-1].date(),
    "BTC above SMA50": bool(latest["btc_above_sma50"]),
    "BTC 20d momentum positive": bool(latest["btc_roc20_pos"]),
    "QQQ above SMA100": bool(latest["qqq_above_sma100"]),
    "QQQ 20d return": latest["qqq_roc20"],
    "Gold 20d return": latest["gold_roc20"],
    "Dollar proxy 20d return": latest["dollar_roc20"],
    "Long bond 20d return": latest["long_bond_roc20"],
    "Short-term anti-fiat divergence": bool(short_term_anti_fiat.iloc[-1]),
    "Regime model QQQ weight": weights["Regime score"].iloc[-1],
})

## Takeaways

1. 跨资产模型的主要贡献是降低波动和留出样本回撤，不是追求比QQQ更高的绝对收益。
2. 2023—2026留出样本明显优于原始BTC开关，但2020—2022验证期明显落后，说明因子权重存在状态依赖。
3. 短期反法币背离之后，历史QQQ平均收益没有变差；直接压低仓位的增量也很小，因此只保留状态标签，不纳入正式交易规则。
4. 成本提高到单边20个基点时，留出样本中评分模型仍优于BTC开关，但绝对收益明显下降；高换手仍是需要优化的问题。
5. Bootstrap中评分模型留出样本夏普高于原策略的概率约89%，但90%区间仍跨过零，证据还没强到可以宣布稳定胜出。
6. 下一轮应做滚动训练、阈值稳定性和真实开盘执行测试；在这些检查完成前，不把当前参数视为实盘建议。